# pq-kyber-chacha20-poly1305 — interactive demo
ML-KEM-768 (Kyber, FIPS 203) → HKDF-SHA256 → ChaCha20-Poly1305

Run the cells from top to bottom (**Runtime → Run all**). Nothing is stored or sent anywhere. Educational demo only.

In [ ]:
!pip install -q kyber-py cryptography
!git clone -q https://github.com/shahdxf/pq-kyber-chacha20-poly1305.git 2>/dev/null || true
import sys; sys.path.insert(0, "pq-kyber-chacha20-poly1305/src")
from dataclasses import replace
from pq_kyber_chacha import CIPHER_NAME, DecryptionError, Envelope, decrypt, encrypt, generate_keypair
print("ready:", CIPHER_NAME)

## 1. The recipient generates a key pair

In [ ]:
public_key, private_key = generate_keypair()
print("public key :", len(public_key), "bytes  (safe to share)")
print("private key:", len(private_key), "bytes  (never shared)")

## 2. Encrypt a message

In [ ]:
message = "Hello, post-quantum world"  #@param {type:"string"}
context = "conversation-42"  #@param {type:"string"}
env = encrypt(public_key, message.encode(), aad=context.encode())
wire = env.to_bytes()
print(f"envelope = {len(wire)} bytes (KEM ciphertext {len(env.kem_ciphertext)} B)")
print("ciphertext:", env.ciphertext.hex()[:48], "...")

## 3. Decrypt with the right key and the same context

In [ ]:
print(decrypt(private_key, wire, aad=context.encode()).decode())

## 4. What must fail

In [ ]:
def attempt(label, fn):
    try:
        fn(); print(f"{label:28} -> ACCEPTED (bad!)")
    except Exception as e:
        print(f"{label:28} -> rejected ({type(e).__name__})")

_, other_private_key = generate_keypair()
attempt("wrong private key", lambda: decrypt(other_private_key, wire, aad=context.encode()))
attempt("different context", lambda: decrypt(private_key, wire, aad=b"conversation-43"))
for name, pos in [("flip a header bit", 4), ("flip a KEM ciphertext bit", 100), ("flip a nonce bit", 1093), ("flip a ciphertext bit", 1105), ("flip a tag bit", len(wire) - 1)]:
    bad = bytearray(wire); bad[pos] ^= 1
    attempt(name, lambda bad=bad: decrypt(private_key, bytes(bad), aad=context.encode()))